In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import re
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider

In [2]:
def parse_time(log_path):
    """Parse execution time from the log file."""
    with open(log_path, 'r') as f:
        content = f.read()
        match = re.search(r'real\s+(\d+)m(\d+\.\d+)s', content)
        if match:
            minutes = int(match.group(1))
            seconds = float(match.group(2))
            return minutes * 60 + seconds
        else:
            return None

def get_energy(energy_path):
    """Get total energy consumed from energy.csv."""
    df = pd.read_csv(energy_path)
    # Assuming scope 1 is the total energy scope
    scope1_values = df[df['scope'] == 1]['value']
    if not scope1_values.empty:
        last_value = scope1_values.iloc[-1]
        first_value = scope1_values.iloc[0]
        return last_value - first_value
    return None

def get_pcap_mean(pcap_path):
    """Get the mean PCAP value from PCAP_file.csv."""
    df = pd.read_csv(pcap_path)
    return df['value'].mean()

def pareto_front(points):
    """Compute the Pareto front for minimization of both objectives."""
    # Sort points by execution time ascending
    points.sort(key=lambda x: x[0])
    front = []
    min_energy = float('inf')
    for t, e in points:
        if e < min_energy:
            min_energy = e
            front.append((t, e))
    return front

In [3]:
# Directory containing the data
data_dir = '/Users/akhileshraj/Documents/summer2024/main_codes/experiment_data/plotting_data_ISORC/ones-stream-full'

# Collect all points
raw_points = []
for item in os.listdir(data_dir):
    if item.startswith('compressed_iteration_') and os.path.isdir(os.path.join(data_dir, item)):
        log_path = os.path.join(data_dir, item, 'ones-stream-full_output.log')
        energy_path = os.path.join(data_dir, item, 'energy.csv')
        pcap_path = os.path.join(data_dir, item, 'PCAP_file.csv')
        if os.path.exists(log_path) and os.path.exists(energy_path) and os.path.exists(pcap_path):
            time_s = parse_time(log_path)
            energy_j = get_energy(energy_path)
            pcap_mean = get_pcap_mean(pcap_path)
            if time_s is not None and energy_j is not None and pcap_mean is not None:
                raw_points.append((time_s, energy_j, pcap_mean))

# Group by PCAP mean and average
from collections import defaultdict
pcap_groups = defaultdict(list)
for t, e, p in raw_points:
    pcap_groups[p].append((t, e))

# Average within each PCAP group and compute std
points = []
errors = []
for p, group in pcap_groups.items():
    ts = [t for t, e in group]
    es = [e for t, e in group]
    avg_t = np.mean(ts)
    avg_e = np.mean(es) / 1000  # Energy in kJ
    std_e = np.std(es) / 1000   # Std in kJ
    points.append((avg_t, avg_e))
    errors.append(std_e)

# Sort points by execution time
points.sort(key=lambda x: x[0])

# Compute Pareto front
pareto = pareto_front(points.copy())

In [4]:
# Compute Pareto min and max time
pareto_ts = [p[0] for p in pareto]
pareto_es = [p[1] for p in pareto]
pareto_min_t = min(pareto_ts)
pareto_max_t = max(pareto_ts)

# Create figure
fig = go.Figure()

# Add Pareto front
fig.add_trace(go.Scatter(x=pareto_ts, y=pareto_es, mode='lines+markers', name='Pareto Front'))

# Add selected point
selected_t = pareto_ts[0]
selected_e = pareto_es[0]
fig.add_trace(go.Scatter(x=[selected_t], y=[selected_e], mode='markers', name='Selected', marker=dict(size=10, color='red')))

# Create frames
frames = []
for perf in range(101):
    target_t = pareto_min_t + (pareto_max_t - pareto_min_t) * (100 - perf) / 100
    closest_idx = np.argmin(np.abs(np.array(pareto_ts) - target_t))
    selected_t = pareto_ts[closest_idx]
    selected_e = pareto_es[closest_idx]
    frame_data = [
        go.Scatter(x=pareto_ts, y=pareto_es, mode='lines+markers'),
        go.Scatter(x=[selected_t], y=[selected_e], mode='markers', marker=dict(size=10, color='red'))
    ]
    frames.append(go.Frame(data=frame_data, name=str(perf)))

fig.frames = frames

# Set initial frame
fig.update(frames=frames)

In [5]:
# # Add slider and buttons
# fig.update_layout(
#     title='',  # Remove title
#     xaxis=dict(title='Execution Time (seconds)', title_font=dict(size=18), tickfont=dict(size=14)),
#     yaxis=dict(title='Energy Consumption (kJ)', title_font=dict(size=18), tickfont=dict(size=14)),
#     sliders=[dict(
#         active=20,  # Start at 100%
#         steps=[dict(method='animate', args=[[f.name], dict(mode='immediate', frame=dict(duration=300, redraw=False), transition=dict(duration=0))], label=f.name) for f in fig.frames],
#         currentvalue=dict(prefix='Performance %: '),
#     )]
# )

# # Save to HTML
# fig.write_html('pareto_interactive.html')

# print("Interactive HTML saved as 'pareto_interactive.html'")

# # Display the figure in the notebook
# fig.show()

In [8]:
# Performance degradation plot
# Degradation % relative to the fastest point (min time = 0% degradation)
perf_degradation = [(t - pareto_min_t) / pareto_min_t * 100 for t in pareto_ts]

fig2 = go.Figure()

# Pareto front line
fig2.add_trace(go.Scatter(
    x=pareto_ts,
    y=pareto_es,
    mode='lines+markers',
    name='Pareto Front',
    marker=dict(size=10, color='steelblue'),
    line=dict(color='steelblue', width=2),
    hovertemplate=(
        'Exec Time: %{x:.1f} s<br>'
        'Energy: %{y:.2f} kJ<br>'
        'Perf Degradation: %{text:.1f}%<extra></extra>'
    ),
    text=perf_degradation,
))

# Annotations for each point
annotations = []
for t, e, deg in zip(pareto_ts, pareto_es, perf_degradation):
    annotations.append(dict(
        x=t,
        y=e,
        text=f'{deg:.1f}%',
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=1.5,
        arrowcolor='gray',
        ax=0,
        ay=-30,
        font=dict(size=13, color='darkred'),
        bgcolor='rgba(255,255,255,0.7)',
        bordercolor='gray',
        borderwidth=1,
    ))

fig2.update_layout(
    xaxis=dict(title='Execution Time (seconds)', title_font=dict(size=20), tickfont=dict(size=14)),
    yaxis=dict(title='Energy Consumption (kJ)', title_font=dict(size=20), tickfont=dict(size=14)),
    annotations=annotations,
    legend=dict(font=dict(size=13)),
    title=dict(
        text='Pareto Front — Performance Degradation at Each Point',
        font=dict(size=20),
        x=0.5,
    ),
    margin=dict(l=60, r=20, t=50, b=60),
    autosize=True,
)

fig2.write_html('pareto_degradation.html')
fig2.write_image('pareto_degradation.pdf', format='pdf', width=1200, height=700, scale=2)
print("Saved as 'pareto_degradation.html' and 'pareto_degradation.pdf'")
fig2.show()
print("Saved as 'pareto_degradation.html'")
fig2.show()

Saved as 'pareto_degradation.html' and 'pareto_degradation.pdf'


Saved as 'pareto_degradation.html'


In [7]:
# # Create GIF animation
# import imageio

# images = []
# for frame in fig.frames:
#     fig.update(data=frame.data)
#     img_bytes = fig.to_image(format='png')
#     img = imageio.imread(img_bytes)
#     images.append(img)

# imageio.mimsave('pareto_animation.gif', images, duration=0.3)

# print("GIF animation saved as 'pareto_animation.gif'")